# Credit Scoring Model

This notebook builds a machine learning model to predict whether an applicant is high-risk or low-risk using financial history features.

Objective: Predict an individual's creditworthiness using past financial data.
Approach: Use classification models like Logistic Regression and Random Forest.
Metrics: Precision, Recall, F1-score, ROC-AUC.


In [5]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_dir = Path.cwd()
data_dir = project_dir / 'data'
data_dir.mkdir(exist_ok=True, parents=True)
model_dir = project_dir / 'models'
model_dir.mkdir(exist_ok=True, parents=True)

print('Project folder:', project_dir)


ModuleNotFoundError: No module named 'joblib'

In [ ]:
import numpy as np

def make_dataset(n_samples: int = 5000, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    age = rng.integers(21, 68, size=n_samples)
    income = rng.lognormal(mean=11.1, sigma=0.45, size=n_samples).astype(float)
    income = np.clip(income, 20000, 250000)
    debt_to_income = rng.beta(2.5, 4.5, size=n_samples) * 0.8 + 0.08
    credit_utilization = rng.beta(2.2, 4.8, size=n_samples) * 0.8 + 0.05
    late_payments = rng.poisson(1.25, size=n_samples)
    credit_history_years = rng.integers(1, 18, size=n_samples)
    loan_amount = rng.lognormal(mean=9.8, sigma=0.7, size=n_samples)
    loan_amount = np.clip(loan_amount, 2000, 150000)
    months_employed = rng.integers(3, 180, size=n_samples)
    savings_balance = rng.lognormal(mean=9.5, sigma=0.85, size=n_samples)
    savings_balance = np.clip(savings_balance, 500, 150000)
    existing_loans = rng.poisson(1.8, size=n_samples)
    monthly_debt_payment = rng.gamma(2.8, 150, size=n_samples)
    employment_status = rng.choice(['Full-time', 'Part-time', 'Self-employed', 'Unemployed'], size=n_samples, p=[0.52, 0.18, 0.19, 0.11])

    risk_score = (
        1.2 * (debt_to_income * 100)
        + 1.5 * (credit_utilization * 100)
        + 7.0 * late_payments
        + 0.8 * np.maximum(existing_loans - 1, 0)
        + 0.5 * np.maximum(24 - credit_history_years, 0)
        + 0.03 * np.maximum(0.0, 120000 - income) / 1000
        + 0.05 * np.maximum(0.0, 60000 - savings_balance) / 1000
        - 0.18 * months_employed / 12
        - 0.4 * np.minimum(1, np.log1p(income) / 12)
    )

    risk_probability = 1 / (1 + np.exp(-(risk_score - 55)))
    risk_flag = rng.binomial(1, risk_probability)

    data = pd.DataFrame({
        'age': age,
        'income': income.round(2),
        'debt_to_income_ratio': debt_to_income.round(4),
        'credit_utilization': credit_utilization.round(4),
        'late_payments': late_payments,
        'credit_history_years': credit_history_years,
        'loan_amount': loan_amount.round(2),
        'months_employed': months_employed,
        'savings_balance': savings_balance.round(2),
        'existing_loans': existing_loans,
        'monthly_debt_payment': monthly_debt_payment.round(2),
        'employment_status': employment_status,
        'risk_flag': risk_flag,
    })

    data['credit_score'] = 850 - (data['debt_to_income_ratio'] * 300) - (data['credit_utilization'] * 250) - (data['late_payments'] * 25) + (data['months_employed'] / 12) * 10
    data['credit_score'] = np.clip(data['credit_score'], 300, 850).round(2)
    return data

dataset = make_dataset()
dataset.head()

data_path = data_dir / 'credit_scoring_data.csv'
dataset.to_csv(data_path, index=False)
print(f'Generated {len(dataset)} rows at {data_path}')


Generated 5000 rows at C:\2nd year\rishabh\credit_scoring_model\notebooks\data\credit_scoring_data.csv


In [ ]:
df = pd.read_csv(data_dir / 'credit_scoring_data.csv')
df.info()
df.describe().T


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   5000 non-null   int64  
 1   income                5000 non-null   float64
 2   debt_to_income_ratio  5000 non-null   float64
 3   credit_utilization    5000 non-null   float64
 4   late_payments         5000 non-null   int64  
 5   credit_history_years  5000 non-null   int64  
 6   loan_amount           5000 non-null   float64
 7   months_employed       5000 non-null   int64  
 8   savings_balance       5000 non-null   float64
 9   existing_loans        5000 non-null   int64  
 10  monthly_debt_payment  5000 non-null   float64
 11  employment_status     5000 non-null   str    
 12  risk_flag             5000 non-null   int64  
 13  credit_score          5000 non-null   float64
dtypes: float64(7), int64(6), str(1)
memory usage: 547.0 KB


,count,mean,std,min,25%,50%,75%,max
age,5000.0,43.936400,13.595930,21.0000,32.000000,44.00000,56.0000,67.0000
income,5000.0,73679.488988,34782.266722,20000.0000,49096.940000,66109.12500,90138.2500,250000.0000
debt_to_income_ratio,5000.0,0.364496,0.135497,0.0882,0.260975,0.35275,0.4569,0.8323
credit_utilization,5000.0,0.302161,0.132556,0.0516,0.198300,0.28820,0.3921,0.7872
late_payments,5000.0,1.242600,1.121783,0.0000,0.000000,1.00000,2.0000,6.0000
credit_history_years,5000.0,9.096200,4.883207,1.0000,5.000000,9.00000,13.0000,17.0000
loan_amount,5000.0,23063.206466,17912.488496,2000.0000,11172.702500,18092.12000,29272.6450,150000.0000
months_employed,5000.0,90.337600,51.167525,3.0000,45.000000,90.00000,134.0000,179.0000
savings_balance,5000.0,18825.796180,18745.970261,735.2700,7350.942500,12905.41500,23546.4600,150000.0000
existing_loans,5000.0,1.800600,1.349075,0.0000,1.000000,2.00000,3.0000,9.0000


In [ ]:
TARGET = 'risk_flag'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_train.shape, X_test.shape


((3750, 13), (1250, 13))

In [ ]:
categorical_features = ['employment_status']
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ]
)

models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5, class_weight='balanced', random_state=42),
}

results = []
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probs),
    })

results


[{'model': 'logistic_regression',
  'accuracy': 0.9832,
  'precision': 1.0,
  'recall': 0.9829268292682927,
  'f1': 0.991389913899139,
  'roc_auc': 0.9984959349593496},
 {'model': 'random_forest',
  'accuracy': 0.9832,
  'precision': 0.9967132292522597,
  'recall': 0.9861788617886179,
  'f1': 0.9914180629342052,
  'roc_auc': 0.9926422764227643}]

In [ ]:
best_result = max(results, key=lambda item: item['roc_auc'])
best_name = best_result['model']
best_model = Pipeline([('preprocessor', preprocessor), ('model', models[best_name])])
best_model.fit(X_train, y_train)

joblib.dump(best_model, model_dir / 'best_credit_model.joblib')
with open(model_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'best_model': best_name, 'metrics': best_result}, f, indent=2)

best_result


{'model': 'logistic_regression',
 'accuracy': 0.9832,
 'precision': 1.0,
 'recall': 0.9829268292682927,
 'f1': 0.991389913899139,
 'roc_auc': 0.9984959349593496}

## Prediction Example
Use the saved model to predict risk for a sample applicant.


In [ ]:
sample = pd.DataFrame([{
    'age': 35,
    'income': 82000,
    'debt_to_income_ratio': 0.42,
    'credit_utilization': 0.58,
    'late_payments': 1,
    'credit_history_years': 7,
    'loan_amount': 16000,
    'months_employed': 48,
    'savings_balance': 20000,
    'existing_loans': 2,
    'monthly_debt_payment': 540,
    'employment_status': 'Full-time',
    'credit_score': 690
}])

prediction = best_model.predict(sample)[0]
probability = best_model.predict_proba(sample)[0, 1]

print({
    'prediction': int(prediction),
    'label': 'High risk' if prediction == 1 else 'Low risk',
    'risk_probability': round(float(probability), 4)
})


{'prediction': 1, 'label': 'High risk', 'risk_probability': 1.0}
